# Sales Territory Analysis – East Region (Connecticut & New York)
**Author:** Lorah Mwembe  
**Program:** Year Up United – Data Analytics  

This project analyzes sales performance for the **East Region**, focusing on my two assigned sales territories:

- **Territory 1:** Connecticut  
- **Territory 2:** New York  

Using Python, Pandas, and Matplotlib, I will:
- Identify territory managers and store locations  
- Analyze monthly in‑store revenue  
- Rank store performance  
- Identify top customers  
- Analyze product category trends  
- Provide recommendations for next‑quarter marketing focus  


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # just to make charts look nicer


In [10]:
# Loading customer data
# sep="|" because the file uses pipe separators instead of commas
customers = pd.read_csv("customer_list.csv", sep="|")

# Checking structure: column names, non-null counts, and data types
customers.info()

# Previewing the first few rows to understand the data
customers.head()



<class 'pandas.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   cust_id       521 non-null    int64
 1   date          521 non-null    str  
 2   time          521 non-null    str  
 3   name          521 non-null    str  
 4   email         521 non-null    str  
 5   phone         520 non-null    str  
 6   sms-opt-out   520 non-null    str  
dtypes: int64(1), str(6)
memory usage: 28.6 KB


,cust_id,date,time,name,email,phone,sms-opt-out
0,1,2023-03-15,08:45:12,Rachel,rachel@centralperk.coffee,212-555-1001,N
1,2,2023-05-22,12:30:45,R. Geller,rossg@centralperk.coffee,212-555-1002,N
2,3,2023-07-09,18:15:27,Monica Geller,chefmonica@centralperk.coffee,212-555-1003,N
3,4,2023-09-01,21:05:33,Chandler Bing,chandlerb@centralperk.coffee,212-555-1004,Y
4,5,2023-11-18,14:22:10,Joey,howyoudoing@centralperk.coffee,212-555-1005,N


In [11]:
# Loading product category lookup table
product_categories = pd.read_csv("ProductCategories.csv")
product_categories.info()
product_categories.head()


<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   CategoryID     52 non-null     int64
 1   Category       52 non-null     str  
 2   SubcategoryID  52 non-null     str  
 3   Subcategory    52 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.8 KB


,CategoryID,Category,SubcategoryID,Subcategory
0,120,Technology & Accessories,120-tab,Tablets
1,120,Technology & Accessories,120-cal,Calculators
2,120,Technology & Accessories,120-sof,Software Download
3,120,Technology & Accessories,120-hea,Headphones
4,120,Technology & Accessories,120-ext,External Accessories


In [13]:
store_sales = pd.read_csv("store_sales.csv")
store_sales.info()
store_sales.head()


FileNotFoundError: [Errno 2] No such file or directory: 'store_sales.csv'

In [14]:
# Loading product details
products = pd.read_csv("Products.csv")
products.info()
products.head()

<class 'pandas.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Prod Num       669 non-null    str  
 1   Product        669 non-null    str  
 2   CategoryID     669 non-null    int64
 3   SubcategoryID  669 non-null    str  
dtypes: int64(1), str(3)
memory usage: 21.0 KB


,Prod Num,Product,CategoryID,SubcategoryID
0,105248-IT,TCL NXTPAPER 10s,120,120-tab
1,105249-IT,Dell Latitude 7320 Detachable,120,120-tab
2,105250-IT,Realme Pad,120,120-tab
3,105251-IT,Lenovo Tab P12 Pro,120,120-tab
4,105252-IT,Microsoft Surface Pro 9,120,120-tab


In [ ]:
# Loading store information including state, store ID, territory manager, and region
store_detail = pd.read_csv("StoreDetail.csv")
store_detail.info()
store_detail.head()


5.##Core Marketin Analysis 

In [ ]:
 Q1 – Territory managers + store IDs + cities

### 1. Territory managers and store locations

**Question:**  
Who are the territory managers for the assigned sales territories (Connecticut and New York)?  
What are the store IDs and cities for the stores in each assigned sales territory?


In [ ]:
# Filter to just Connecticut and New York in the East region
east_ct_ny = store_detail[
    (store_detail["Region"] == "East") &
    (store_detail["State"].isin(["Connecticut", "New York"]))
]

east_ct_ny[["State", "Store ID", "Store Location", "Territory Manager"]].sort_values(
    by=["State", "Store ID"]
)



NameError: name 'store_detail' is not defined

### 2. Monthly total revenue for in-store sales (CT vs NY)

**Question:**  
What is monthly total revenue for in-store sales in each of the two sales territories over the full period?


In [ ]:
# Merge sales with store details to get State and Region
sales_with_store = store_sales.merge(
    store_detail,
    on="Store ID",
    how="left"
)

# Filter to East region, CT & NY, and in-store channel
east_sales = sales_with_store[
    (sales_with_store["Region"] == "East") &
    (sales_with_store["State"].isin(["Connecticut", "New York"])) &
    (sales_with_store["Channel"] == "In-Store")  # adjust if your column name differs
]

# Create a Year-Month column for grouping
east_sales["YearMonth"] = pd.to_datetime(east_sales["Date"]).dt.to_period("M")

# Monthly total revenue per state
monthly_revenue = (
    east_sales
    .groupby(["State", "YearMonth"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values(["State", "YearMonth"])
)

monthly_revenue.head()



NameError: name 'store_sales' is not defined

In [ ]:
### Monthly total revenue for in-store sales

To calculate monthly revenue, I first merge sales with store details so I can see which state each store belongs to. Then I filter to in-store sales in Connecticut and New York, create a Year-Month column, and sum revenue by state and month.


### 8.Q3. Store performance ranking

**Question:**  
How would I rank the sales performance of each store in each sales territory?  
Which are the top-performing stores?


In [ ]:
store_performance = (
    east_sales
    .groupby(["State", "Store ID", "Store Location"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values(["State", "Revenue"], ascending=[True, False])
)

store_performance.head(10)



In [ ]:
store_performance["Revenue_Rank"] = (
    store_performance
    .groupby("State")["Revenue"]
    .rank(ascending=False, method="dense")
)

store_performance.head(10)



### 9. Q4- Top customers in each sales territory (customer_id vs rewards_id)

**Question:**  
Comparing the customer ID from the customer list data with the rewards ID from the sales data,  
who were the top customers in each sales territory?


In [ ]:
# Merge sales with customer list
sales_with_customers = east_sales.merge(
    customer_list,
    left_on="Rewards ID",   # adjust to your actual column name
    right_on="cust_id",
    how="left"
)

top_customers = (
    sales_with_customers
    .groupby(["State", "cust_id", "name"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values(["State", "Revenue"], ascending=[True, False])
)

top_customers.head(10)


10. Q5 – Transactions & revenue per month by product category

### 5. Product category performance by month

**Questions:**  
- What is the number of transactions per month by product category in each assigned territory?  
- What is total sales revenue per month by category?  
- What might this tell me about the most popular products and growth opportunities?


In [ ]:
# Merge sales with products
sales_products = east_sales.merge(
    products,
    on="Prod Num",
    how="left"
)

# Merge with product categories
sales_products = sales_products.merge(
    product_categories,
    on=["CategoryID", "SubcategoryID"],
    how="left"
)

# Group by month, state, and category
sales_products["YearMonth"] = pd.to_datetime(sales_products["Date"]).dt.to_period("M")

category_summary = (
    sales_products
    .groupby(["State", "YearMonth", "Category"])  # Category from ProductCategories.csv
    .agg(
        Transactions=("Revenue", "count"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
    .sort_values(["State", "YearMonth", "Total_Revenue"], ascending=[True, True, False])
)

category_summary.head()



### Recommendation for next quarter

Based on my analysis, I recommend focusing marketing efforts on:

- New York stores with consistently high monthly revenue but lower transactions in high-margin categories like Technology & Accessories.
-Connecticut stores where Apparel and Merchandise is growing month-over-month but still underperforms Textbooks.

I would propose targeted promotions (e.g., bundle discounts or loyalty rewards) for these categories in the top-performing stores, while also testing campaigns in underperforming locations to see if we can lift sales in specific product categories.


In [ ]:
12. Matplotlib Charts

In [ ]:
# Top 5 New York stores( x= store, y=revenue)
ny_top5 = (
    store_performance[store_performance["State"] == "New York"]
    .nlargest(5, "Revenue")
)

plt.figure(figsize=(8, 4))
plt.bar(ny_top5["Store Location"], ny_top5["Revenue"], color="steelblue")
plt.title("Top 5 Stores by Revenue – New York")
plt.xlabel("Store Location")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Monthly revenue by state (line chart trend over time)
pivot_monthly = monthly_revenue.pivot(
    index="YearMonth",
    columns="State",
    values="Revenue"
)

pivot_monthly.plot(kind="line", figsize=(8, 4), marker="o")
plt.title("Monthly In-Store Revenue – Connecticut vs New York")
plt.xlabel("Year-Month")
plt.ylabel("Total Revenue")
plt.legend(title="State")
plt.tight_layout()
plt.show()


In [ ]:
import os
os.getcwd()


'C:\\Users\\lorah\\Documents\\DATA_Labs\\Capstone_2'

In [ ]:
.../Documents/DATA_Labs/Capstone_2


NameError: name 'Documents' is not defined

In [ ]:
import os
os.getcwd()


'C:\\Users\\lorah\\Documents\\DATA_Labs\\Capstone_2'

In [ ]:
os.listdir()


['.git',
 '.ipynb_checkpoints',
 'data',
 'mwembe_sales_analysis.ipynb',
 'README.md']

In [ ]:
import os
os.listdir("data")


['.ipynb_checkpoints',
 'customer_list.csv',
 'ProductCategories.csv',
 'Products.csv',
 'StoreDetail.csv',
 'StoreSales.csv',
 'Untitled.ipynb']

In [ ]:
customers = pd.read_csv("customer_list.csv", sep="|")
products = pd.read_csv("Products.csv")
product_categories = pd.read_csv("ProductCategories.csv")
store_detail = pd.read_csv("StoreDetail.csv")
store_sales = pd.read_csv("StoreSales.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'customer_list.csv'